In [131]:
from __future__ import annotations
import json, math, re, unicodedata, uuid, random
from dataclasses import dataclass
from pathlib import Path
from datetime import date, datetime, time, timedelta
import re
import pandas as pd
import numpy as np

In [132]:
PY_SEED = 42
np.random.seed(PY_SEED)
random.seed(PY_SEED)

YEAR = 2025

FECHA_INI = date(YEAR, 1, 1)
FECHA_FIN = date(YEAR, 7, 31)

N_EMPRESAS_TRANSP = 3
USUARIOS_POR_EMPRESA = 3
TICKETS_POR_USUARIO = 50    
PESO_TARJETA = 0.85          
IVA_TIPO = 0.21
IMP_ELEC = 0.051 

STATION_OFFSET_MIN, STATION_OFFSET_MAX = -0.02, 0.02  # €/L
DAILY_NOISE_MIN, DAILY_NOISE_MAX       = -0.01, 0.01  # €/L

LITROS_MIN, LITROS_MAX, LITROS_MODA = 10.0, 80.0, 35.0

PRODUCTOS = [
    "Gasolina 95 E5",
    "Gasolina 98 E5",
    "Gasóleo A",
    "Gasóleo Premium",
]

DATA_DIR = Path(r"data")
PATH_PUNTOS = DATA_DIR / "PuntosCarga.csv"
PATH_CIFS   = DATA_DIR / "CIFs_puntos_carga.csv"
PATH_PRECIOS= DATA_DIR / "ev_table_ACDC_2025.csv"

OUT_JSONL = DATA_DIR / "tickets_sinteticos.jsonl"
OUT_JSON  = DATA_DIR / "tickets_sinteticos.json"

In [133]:
def prov_key(x: str) -> str:
    s = norm_txt(x)
    if "/" in s: s = s.split("/")[0].strip()
    repl = {
        "vizcaya":"bizkaia",
        "guipuzcoa":"gipuzkoa",
        "guipúzcoa":"gipuzkoa",
        "la coruna":"a coruna",
        "coruna":"a coruna",
        "coruña":"a coruna",
        "orense":"ourense",
        "lerida":"lleida",
        "gerona":"girona",
        "valencia/valencia":"valencia",
        "alicante/alacant":"alicante",
        "castello":"castellon",
        "tenerife":"santa cruz de tenerife",
        "araba alava":"alava",
        "araba/alava":"alava",
    }
    return repl.get(s, s)

In [134]:
from pathlib import Path
import pandas as pd

def load_csv_guess(path: Path):
    encodings = ["cp1252", "utf-8", "latin1"]
    seps = [",", ";", "|", "\t"]
    last_err = None
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                if df.shape[1] >= 1:
                    return df
            except Exception as e:
                last_err = e
                continue
    raise RuntimeError(f"No pude leer {path} — último error: {last_err}")

In [135]:
def norm_txt(x: str) -> str:
    if pd.isna(x): 
        return ""
    x = str(x).strip()
    x = "".join(c for c in unicodedata.normalize("NFD", x) if unicodedata.category(c) != "Mn")
    return x.lower()

def round2(x: float) -> float:
    return float(np.round(x + 1e-12, 2))

def round3(x: float) -> float:
    return float(np.round(x + 1e-12, 3))

def random_fecha(fecha_ini: date, fecha_fin: date) -> date:
    delta = (fecha_fin - fecha_ini).days
    return fecha_ini + timedelta(days=int(np.random.randint(0, delta + 1)))

def random_hora() -> time:
    return (datetime.min + timedelta(seconds=int(np.random.randint(0, 24*3600)))).time()

def elegir_metodo_pago() -> str:
    return "Tarjeta credito" if random.random() < PESO_TARJETA else "Efectivo"

def str_fecha(d: date) -> str:
    return d.strftime("%Y-%m-%d")

def str_hora(t: time) -> str:
    return t.strftime("%H:%M:%S")

In [136]:
# Bloque 3 — Carga Puntos de Carga (reemplazo robusto provincia)
import re

raw_pc = load_csv_guess(PATH_PUNTOS).copy()
pc = raw_pc.copy()
pc.columns = [norm_txt(c) for c in pc.columns]

def _simplify(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", norm_txt(s))

print("Columnas detectadas en PuntosCarga (normalizadas):", list(pc.columns))

for cand in ["id","id_punto","id_estacion","codigo","idpunto","ideess","id_eess"]:
    if cand in pc.columns:
        col_id = cand; break
else:
    col_id = "_idgen"
    pc[col_id] = np.arange(1, len(pc)+1)

for cand in ["nombre","estacion","punto","rotulo","rótulo","site","ubicacion","ubicación"]:
    if cand in pc.columns:
        col_nombre = cand; break
else:
    col_nombre = None

for cand in ["operador","empresa","grupo","compania","compañia","marca","rotulo","rótulo"]:
    if cand in pc.columns:
        col_grupo = cand; break
else:
    col_grupo = None

col_prov = None
for cand in ["provincia","province","prov","provincia_nombre","desc_provincia","provinciaev","provincia_ev"]:
    if cand in pc.columns:
        col_prov = cand; break
if not col_prov:
    for c in pc.columns:
        sc = _simplify(c)
        if ("provincia" in sc) or (sc == "province") or (sc.startswith("prov") and not sc.startswith("proveedor")):
            col_prov = c; break

if not col_prov:
    CP_TO_PROV = {
        "01":"Araba/Álava","02":"Albacete","03":"Alicante","04":"Almería","05":"Ávila","06":"Badajoz",
        "07":"Illes Balears","08":"Barcelona","09":"Burgos","10":"Cáceres","11":"Cádiz","12":"Castellón",
        "13":"Ciudad Real","14":"Córdoba","15":"A Coruña","16":"Cuenca","17":"Girona","18":"Granada",
        "19":"Guadalajara","20":"Gipuzkoa","21":"Huelva","22":"Huesca","23":"Jaén","24":"León",
        "25":"Lleida","26":"La Rioja","27":"Lugo","28":"Madrid","29":"Málaga","30":"Murcia",
        "31":"Navarra","32":"Ourense","33":"Asturias","34":"Palencia","35":"Las Palmas","36":"Pontevedra",
        "37":"Salamanca","38":"Santa Cruz de Tenerife","39":"Cantabria","40":"Segovia","41":"Sevilla",
        "42":"Soria","43":"Tarragona","44":"Teruel","45":"Toledo","46":"València/Valencia","47":"Valladolid",
        "48":"Bizkaia","49":"Zamora","50":"Zaragoza","51":"Ceuta","52":"Melilla"
    }
    def cp_a_prov(x):
        s = re.sub(r"[^0-9]","", str(x))
        if len(s) < 2: return None
        return CP_TO_PROV.get(s[:2].zfill(2))
    col_cp = None
    for cand in ["cp","c_p","codigo_postal","codigopostal","postal","zip","c. p.","cod_postal","cpostal"]:
        if cand in pc.columns:
            col_cp = cand; break
    if col_cp:
        pc["_provincia_fill"] = pc[col_cp].map(cp_a_prov)
        if pc["_provincia_fill"].notna().any():
            col_prov = "_provincia_fill"

if not col_prov:
    raise ValueError("No encuentro columna de provincia en PuntosCarga.csv (ni derivable por C.P.)")

for cand in ["municipio","localidad","poblacion","ciudad","town"]:
    if cand in pc.columns:
        col_muni = cand; break
else:
    col_muni = None

for cand in ["direccion","dirección","address","dir","calle","via"]:
    if cand in pc.columns:
        col_dir = cand; break
else:
    col_dir = None

col_lat = None
col_lon = None
for c in pc.columns:
    sc = _simplify(c)
    if col_lat is None and ("latitud" in sc or sc.endswith("lat") or sc == "lat"):
        col_lat = c
    if col_lon is None and ("longitud" in sc or sc.endswith("lon") or sc in ("lon","long","lng")):
        col_lon = c

def to_float_locale(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")

lat_series = to_float_locale(pc[col_lat]) if col_lat else pd.Series(np.nan, index=pc.index)
lon_series = to_float_locale(pc[col_lon]) if col_lon else pd.Series(np.nan, index=pc.index)
if (lat_series.lt(-20).mean() > 0.2) or (lon_series.gt(20).mean() > 0.2):
    lat_series, lon_series = lon_series, lat_series

col_pot = None
for cand in ["potencia","potencia_kw","kw","power","power_kw","kw_potencia"]:
    if cand in pc.columns:
        col_pot = cand; break
if col_pot is None:
    pot_candidates = [c for c in pc.columns if "kw" in _simplify(c) or "potencia" in _simplify(c) or "power" in _simplify(c)]
    col_pot = pot_candidates[0] if pot_candidates else None
pot_series = pd.to_numeric(pc[col_pot].astype(str).str.replace(",", ".", regex=False), errors="coerce") if col_pot else pd.Series(np.nan, index=pc.index)

pc_std = pd.DataFrame({
    "id_estacion": pc[col_id].astype(str),
    "nombre": pc[col_nombre].astype(str) if col_nombre else "",
    "grupo": pc[col_grupo].astype(str) if col_grupo else "",
    "provincia": pc[col_prov].astype(str),
    "municipio": pc[col_muni].astype(str) if col_muni else "",
    "direccion": pc[col_dir].astype(str) if col_dir else "",
    "lat": lat_series,
    "lon": lon_series,
    "potencia_kW": pot_series
})
pc_std["provincia_norm"] = pc_std["provincia"].map(prov_key)
pc_std["grupo_norm"] = pc_std["grupo"].map(norm_txt)

if pc_std["potencia_kW"].notna().mean() > 0.8:
    pc_std = pc_std.dropna(subset=["potencia_kW"])
if pc_std["lat"].notna().mean() > 0.8 and pc_std["lon"].notna().mean() > 0.8:
    pc_std = pc_std.dropna(subset=["lat","lon"])

pc_std = pc_std.reset_index(drop=True)
print("Puntos de carga cargados:", len(pc_std))
print(pc_std.head(3))


Columnas detectadas en PuntosCarga (normalizadas): ['idpuntorecarga|operador|direccion|codpostal|provincia|municipio|localizacion|coordenadaxdec|coordenadaydec|potenciamaxima', 'unnamed: 1']
Puntos de carga cargados: 12459
  id_estacion nombre grupo                                          provincia  \
0           1               ES*IBD*E234836|IBERDROLA CLIENTES S.A.U|Sierra...   
1           2               ES*IBD*E235992|IBERDROLA CLIENTES S.A.U|Alquer...   
2           3               ES*IBD*E250412|IBERDROLA CLIENTES S.A.U|Retuer...   

  municipio direccion  lat  lon  potencia_kW  \
0                      NaN  NaN          NaN   
1                      NaN  NaN          NaN   
2                      NaN  NaN          NaN   

                                      provincia_norm grupo_norm  
0  es*ibd*e234836|iberdrola clientes s.a.u|sierra...             
1  es*ibd*e235992|iberdrola clientes s.a.u|alquer...             
2  es*ibd*e250412|iberdrola clientes s.a.u|retuer...         

In [137]:
raw_cifs = load_csv_guess(PATH_CIFS).copy()
cifs = raw_cifs.copy()
cifs.columns = [norm_txt(c) for c in cifs.columns]

def simplify_colname(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", norm_txt(s))

cols = list(cifs.columns)
cols_simpl = {c: simplify_colname(c) for c in cols}

company_keys = ["compania","razonsocial","empresa","grupo","operador","sociedad","razon","marca","rotulo"]
nif_keys     = ["nif","cif","nifcif","cifnif"]

def pick_col(keys):
    for c in cols:
        sc = cols_simpl[c]
        if any(k in sc for k in keys):
            return c
    return None

col_emp = pick_col(company_keys)
col_nif = pick_col(nif_keys)
if not (col_emp and col_nif):
    raise ValueError("CIFs_puntos_carga.csv debe tener columnas de compañía/grupo y NIF/CIF")

map_grupo_to_cif = {}
for _, r in cifs.iterrows():
    emp = r[col_emp]
    cif = r[col_nif]
    if pd.isna(emp) or pd.isna(cif): continue
    k = norm_txt(emp)
    map_grupo_to_cif[k] = str(cif).strip().upper()

In [138]:
PROV_TO_CCAA = {
    "almeria":"andalucia","cadiz":"andalucia","cordoba":"andalucia","granada":"andalucia",
    "huelva":"andalucia","jaen":"andalucia","malaga":"andalucia","sevilla":"andalucia",
    "huesca":"aragon","teruel":"aragon","zaragoza":"aragon",
    "asturias":"asturias",
    "illes balears":"baleares","baleares":"baleares","islas baleares":"baleares",
    "las palmas":"canarias","santa cruz de tenerife":"canarias",
    "cantabria":"cantabria",
    "albacete":"castilla-la mancha","ciudad real":"castilla-la mancha","cuenca":"castilla-la mancha",
    "guadalajara":"castilla-la mancha","toledo":"castilla-la mancha",
    "avila":"castilla y leon","burgos":"castilla y leon","leon":"castilla y leon","palencia":"castilla y leon",
    "salamanca":"castilla y leon","segovia":"castilla y leon","soria":"castilla y leon",
    "valladolid":"castilla y leon","zamora":"castilla y leon",
    "barcelona":"cataluna","girona":"cataluna","lleida":"cataluna","tarragona":"cataluna",
    "alicante":"comunidad valenciana","castellon":"comunidad valenciana","valencia":"comunidad valenciana",
    "badajoz":"extremadura","caceres":"extremadura",
    "a coruna":"galicia","lugo":"galicia","ourense":"galicia","pontevedra":"galicia",
    "madrid":"madrid",
    "murcia":"murcia",
    "navarra":"navarra",
    "alava":"pais vasco","araba":"pais vasco","gipuzkoa":"pais vasco","bizkaia":"pais vasco",
    "la rioja":"la rioja",
    "ceuta":"ceuta","melilla":"melilla",
}

def prov_to_ccaa(prov_norm: str) -> str:
    return PROV_TO_CCAA.get(prov_norm, prov_norm)

pc_std["ccaa_norm"] = pc_std["provincia_norm"].map(prov_to_ccaa)


In [139]:
# Bloque 7-compat — precio_kwh y nacional (usar tras cargar precios_ev_idx)
from collections import defaultdict

try:
    PROV_TO_CCAA
except NameError:
    PROV_TO_CCAA = {}

def prov_to_ccaa(prov_norm: str) -> str:
    return PROV_TO_CCAA.get(prov_norm, prov_norm)

tmp = defaultdict(list)
for (ccaa, t, m), v in precios_ev_idx.items():
    if v is not None and not pd.isna(v):
        tmp[(t, m)].append(float(v))
precios_ev_nac_idx = {k: float(np.mean(vals)) for k, vals in tmp.items() if vals}

def station_offset_ev(id_estacion: str) -> float:
    rnd = random.Random(hash(id_estacion) & 0xffffffff)
    return float(rnd.uniform(STATION_OFFSET_MIN, STATION_OFFSET_MAX))

def precio_kwh(prov_norm: str, tipo: str, fecha: date, id_estacion: str) -> float:
    ccaa = prov_to_ccaa(prov_norm)
    base = precios_ev_idx.get((ccaa, tipo, fecha.month))
    if base is None:
        base = precios_ev_nac_idx.get((tipo, fecha.month))
    if base is None:
        base = 0.35 if tipo == "AC" else 0.45
    return round3(max(0.05, float(base) + station_offset_ev(id_estacion)))

In [140]:
raw_ev = load_csv_guess(PATH_PRECIOS).copy()
ev = raw_ev.copy()
ev.columns = [norm_txt(c) for c in ev.columns]

def ccaa_key(x: str) -> str:
    s = norm_txt(x)
    repl = {
        "catalunya":"cataluna",
        "cataluña":"cataluna",
        "comunitat valenciana":"comunidad valenciana",
        "c. valenciana":"comunidad valenciana",
        "illes balears":"baleares",
        "islas baleares":"baleares",
        "pais vasco":"pais vasco",
        "euskadi":"pais vasco",
        "comunidad de madrid":"madrid",
        "comunidad foral de navarra":"navarra",
        "region de murcia":"murcia",
        "castilla-la mancha":"castilla-la mancha",
        "castilla la mancha":"castilla-la mancha",
        "castilla y leon":"castilla y leon",
        "aragon":"aragon",
        "andalucia":"andalucia",
        "extremadura":"extremadura",
        "galicia":"galicia",
        "asturias":"asturias",
        "cantabria":"cantabria",
        "la rioja":"la rioja",
        "canarias":"canarias",
        "ceuta":"ceuta",
        "melilla":"melilla",
    }
    return repl.get(s, s)

col_ccaa = None
for cand in ["ccaa","comunidad","comunidad_autonoma","comunidad autonoma","autonomia","region"]:
    if cand in ev.columns:
        col_ccaa = cand; break
if not col_ccaa:
    for c in ev.columns:
        if "ccaa" in c or "comunidad" in c or "autonoma" in c or "autónoma" in c:
            col_ccaa = c; break
if not col_ccaa:
    if not pd.api.types.is_numeric_dtype(ev.iloc[:,0]):
        col_ccaa = ev.columns[0]
if not col_ccaa:
    raise ValueError("No encuentro columna de CCAA en ev_table_ACDC_2025.csv")

pat = re.compile(rf"^{YEAR}-(\d{{2}})_(ac|dc)$", re.IGNORECASE)
colmap = {}  
for c in ev.columns:
    m = pat.match(c)
    if m:
        mes = int(m.group(1))
        tipo = m.group(2).upper()
        if 1 <= mes <= 12 and tipo in ("AC","DC"):
            colmap[c] = (mes, tipo)

if not colmap:
    raise ValueError("No detecto columnas 'YYYY-MM_AC/DC' en ev_table_ACDC_2025.csv")

tmp = ev[[col_ccaa] + list(colmap.keys())].copy()
long = tmp.melt(id_vars=[col_ccaa], var_name="_col", value_name="_precio")
long[["_mes","_tipo"]] = long["_col"].map(lambda s: pd.Series(colmap[str(s)])).apply(pd.Series)
long["_ccaa_norm"] = long[col_ccaa].map(ccaa_key)
long["_precio"] = pd.to_numeric(long["_precio"], errors="coerce")

parsed_ev = long.dropna(subset=["_ccaa_norm","_precio"])
parsed_ev = parsed_ev.loc[parsed_ev["_mes"].isin(range(1,8))].copy()
parsed_ev["_mes"] = parsed_ev["_mes"].astype(int)

precios_ev_idx = parsed_ev.set_index(["_ccaa_norm","_tipo","_mes"])["_precio"].to_dict()

In [141]:
CONTROL_LETTERS = "JABCDEFGHI"
LETTER_GROUP = set(list("PQRSNW"))
DIGIT_GROUP  = set(list("ABEH"))
ANY_GROUP    = set(list("CDFGJUVXYZ"))

def _sum_digits(n: int) -> int:
    return n if n < 10 else n//10 + n%10

def cif_generate() -> str:
    first = random.choice(list(LETTER_GROUP | DIGIT_GROUP | ANY_GROUP))
    digits = [random.randint(0,9) for _ in range(7)]
    sum_even = digits[1] + digits[3] + digits[5]
    sum_odd = sum(_sum_digits(2*d) for d in (digits[0], digits[2], digits[4], digits[6]))
    total = sum_even + sum_odd
    cd_num = (10 - (total % 10)) % 10
    if first in LETTER_GROUP:
        control = CONTROL_LETTERS[cd_num]
    elif first in DIGIT_GROUP:
        control = str(cd_num)
    else:
        control = str(cd_num)
    body = "".join(str(d) for d in digits)
    return f"{first}{body}{control}"

def cif_de_grupo(grupo_texto: str) -> str:
    key = norm_txt(grupo_texto)
    v = map_grupo_to_cif.get(key)
    if isinstance(v, str) and len(v.strip()) >= 8:
        return v.strip().upper()
    return cif_generate()

In [142]:
def station_offset_ev(id_estacion: str) -> float:
    rnd = random.Random(hash(id_estacion) & 0xffffffff)
    return float(rnd.uniform(STATION_OFFSET_MIN, STATION_OFFSET_MAX))

def precio_kwh(prov_norm: str, tipo: str, fecha: date, id_estacion: str) -> float:
    base = precios_ev_idx.get((prov_norm, tipo, fecha.month))
    if base is None:
        base = precios_ev_nac_idx.get((tipo, fecha.month))
    if base is None:
        base = 0.35 if tipo == "AC" else 0.45
    return round3(max(0.05, float(base) + station_offset_ev(id_estacion)))

def calcular_importes_ev(kwh: float, precio_unit: float) -> dict:
    importe_energia = round2(kwh * precio_unit)
    impuesto_electricidad = round2(importe_energia * IMP_ELEC)
    base_iva = round2(importe_energia + impuesto_electricidad)
    iva = round2(base_iva * IVA_TIPO)
    total = round2(base_iva + iva)
    return {
        "precio_kwh": precio_unit,
        "energia_kwh": kwh,
        "importe_energia": importe_energia,
        "impuesto_electricidad": impuesto_electricidad,
        "base_iva": base_iva,
        "iva": iva,
        "total": total
    }

In [143]:
@dataclass
class Empresa:
    id: str
    nombre: str

@dataclass
class Usuario:
    id: str
    empresa_id: str
    nombre: str

empresas = [Empresa(id=f"EMP{i+1:03d}", nombre=f"TransporteEV_{i+1:02d} S.L.") for i in range(N_EMPRESAS_TRANSP)]
usuarios = []
for e in empresas:
    for j in range(USUARIOS_POR_EMPRESA):
        usuarios.append(Usuario(id=f"{e.id}-U{j+1:03d}", empresa_id=e.id, nombre=f"Usuario_{j+1:02d}_{e.id}"))

In [144]:
EV_POOL = pc_std.reset_index(drop=True).copy()
EV_POOL["lat"] = pd.to_numeric(EV_POOL["lat"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
EV_POOL["lon"] = pd.to_numeric(EV_POOL["lon"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
EV_POOL["potencia_kW"] = pd.to_numeric(EV_POOL["potencia_kW"], errors="coerce")
assert len(EV_POOL) > 0

In [145]:
# Bloque 10 — Generador de sesión EV (corregido: sobrescribe si la “provincia” trae blob con '|')

import re
import numpy as np
import pandas as pd

def _coerce_float(x):
    s = str(x).strip().replace(",", ".")
    try:
        return float(s)
    except:
        return np.nan

def _is_lat(v): return 27.0 <= v <= 44.5
def _is_lon(v): return -20.0 <= v <= 5.5
def _looks_blob(v): return isinstance(v, str) and ("|" in v)

def _unpack_pipe_blob(blob: str) -> dict:
    s = str(blob)
    if "|" not in s:
        return {}
    parts = [p.strip() for p in s.split("|")]
    out = {
        "nombre":"", "grupo":"", "direccion":"", "cp":"", "provincia":"", "municipio":"",
        "lat":np.nan, "lon":np.nan, "potencia_kW":np.nan
    }
    if len(parts) >= 9:
        out["grupo"]     = parts[1] if len(parts) > 1 else ""
        out["direccion"] = parts[2] if len(parts) > 2 else ""
        out["cp"]        = parts[3] if len(parts) > 3 else ""
        out["provincia"] = parts[4] if len(parts) > 4 else ""
        out["municipio"] = parts[5] if len(parts) > 5 else ""
        out["nombre"]    = parts[6] if len(parts) > 6 else ""
        # números de cola (lon, lat, potencia)
        cand_nums = []
        for i in range(len(parts)-1, -1, -1):
            v = _coerce_float(parts[i])
            if not np.isnan(v):
                cand_nums.append((i, v))
        # potencia
        for i, v in cand_nums:
            if 1.0 <= v <= 1000.0:
                out["potencia_kW"] = v
                break
        # lat/lon coherentes
        lat, lon = np.nan, np.nan
        for i1, v1 in cand_nums:
            if _is_lat(v1):
                for i2, v2 in cand_nums:
                    if i2 < i1 and _is_lon(v2):
                        lat, lon = v1, v2
                        break
            if not np.isnan(lat) and not np.isnan(lon):
                break
        out["lat"] = lat
        out["lon"] = lon
    else:
        # fallback
        for p in parts:
            v = _coerce_float(p)
            if _is_lat(v): out["lat"] = v
            if _is_lon(v): out["lon"] = v
            if 1.0 <= v <= 1000.0 and np.isnan(out["potencia_kW"]): out["potencia_kW"] = v
        for p in parts:
            if re.fullmatch(r"\d{5}", p):
                out["cp"] = p; break
    for k in ("provincia","municipio","direccion","nombre","grupo"):
        out[k] = str(out[k]).strip()
    return out

def _merge_estacion_row(row) -> dict:
    est = {
        "id": str(row.get("id_estacion", "")),
        "nombre": str(row.get("nombre", "") or "").strip(),
        "grupo": str(row.get("grupo", "") or "").strip(),
        "provincia": str(row.get("provincia", "") or "").strip(),
        "municipio": str(row.get("municipio", "") or "").strip(),
        "direccion": str(row.get("direccion", "") or "").strip(),
        "lat": _coerce_float(row.get("lat", np.nan)),
        "lon": _coerce_float(row.get("lon", np.nan)),
        "potencia_kW": _coerce_float(row.get("potencia_kW", np.nan)),
    }

    # Si cualquier campo viene con blob, desempaqueta y SOBRESCRIBE ese campo
    sources = {
        "provincia": row.get("provincia", ""),
        "municipio": row.get("municipio", ""),
        "direccion": row.get("direccion", ""),
        "nombre":    row.get("nombre", ""),
        "grupo":     row.get("grupo", ""),
    }
    blobs = [v for v in sources.values() if _looks_blob(v)]
    for b in blobs:
        u = _unpack_pipe_blob(b)
        # sobrescribe si el valor actual está vacío o es un blob
        for k in ["nombre","grupo","provincia","municipio","direccion"]:
            if u.get(k):
                if (not est[k]) or _looks_blob(est[k]):
                    est[k] = u[k]
        # coords/potencia: sobrescribe si NaN
        if not np.isnan(u.get("lat", np.nan)) and (pd.isna(est["lat"]) or not _is_lat(est["lat"])):
            est["lat"] = float(u["lat"])
        if not np.isnan(u.get("lon", np.nan)) and (pd.isna(est["lon"]) or not _is_lon(est["lon"])):
            est["lon"] = float(u["lon"])
        if not np.isnan(u.get("potencia_kW", np.nan)) and pd.isna(est["potencia_kW"]):
            est["potencia_kW"] = float(u["potencia_kW"])

    return est

def elegir_duracion_y_kwh(potencia_kW: float) -> tuple[int,float]:
    if pd.isna(potencia_kW) or potencia_kW <= 0:
        potencia_kW = 11.0
    if potencia_kW <= 22:
        dur_min = int(np.random.triangular(20, 60, 180))
    else:
        dur_min = int(np.random.triangular(5, 20, 60))
    uso = float(np.random.triangular(0.5, 0.8, 0.95))
    kwh = round2(max(1.5, potencia_kW * (dur_min/60.0) * uso))
    return dur_min, kwh

def generar_ticket_ev(empresa: Empresa, usuario: Usuario) -> dict:
    f = random_fecha(FECHA_INI, FECHA_FIN)
    h = random_hora()
    row = EV_POOL.sample(1).iloc[0]

    est = _merge_estacion_row(row)
    potencia = est["potencia_kW"] if not pd.isna(est["potencia_kW"]) else 11.0
    tipo = "AC" if potencia <= 22 else "DC"
    dur_min, kwh = elegir_duracion_y_kwh(potencia)

    prov_norm = prov_key(est["provincia"])
    punit = precio_kwh(prov_norm, tipo, f, est["id"])
    imp = calcular_importes_ev(kwh, punit)

    ticket = {
        "idTicket": f"EV-{uuid.uuid4().hex[:12].upper()}",
        "idEmpresa": empresa.id,
        "empresaNombre": empresa.nombre,
        "idUsuario": usuario.id,
        "fechaEmision": str_fecha(f),
        "horaEmision": str_hora(h),
        "metodoPago": elegir_metodo_pago(),
        "estacion": {
            "id": est["id"],
            "nombre": est["nombre"],
            "provincia": est["provincia"],
            "municipio": est["municipio"],
            "direccion": est["direccion"],
            "lat": None if pd.isna(est["lat"]) else float(est["lat"]),
            "lon": None if pd.isna(est["lon"]) else float(est["lon"]),
            "grupo": est["grupo"],
            "nifEmpresa": cif_de_grupo(est["grupo"]),
            "potencia_kW": float(potencia),
            "tipo_carga": tipo
        },
        "carga": {
            "tiempo_min": dur_min,
            "energia_kWh": imp["energia_kwh"],
            "precio_kWh": imp["precio_kwh"]
        },
        "lineas": [
            {
                "concepto": f"Carga {tipo}",
                "energia_kWh": imp["energia_kwh"],
                "precioUnitario_kWh": imp["precio_kwh"],
                "importeEnergia": imp["importe_energia"]
            }
        ],
        "baseImponibleEnergia": imp["importe_energia"],
        "impuestoElectricidad": imp["impuesto_electricidad"],
        "baseIVA": imp["base_iva"],
        "iva": imp["iva"],
        "total": imp["total"],
        "moneda": "EUR",
        "tipoDocumento": "Factura simplificada"
    }
    return ticket


In [146]:
def generar_todos_ev() -> list[dict]:
    tickets = []
    for emp in empresas:
        us_emp = [u for u in usuarios if u.empresa_id == emp.id]
        for u in us_emp:
            for _ in range(TICKETS_POR_USUARIO):
                tickets.append(generar_ticket_ev(emp, u))
    return tickets

tickets_ev = generar_todos_ev()

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for tk in tickets_ev:
        f.write(json.dumps(tk, ensure_ascii=False) + "\n")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(tickets_ev, f, ensure_ascii=False, indent=2)

In [147]:
from pandas import json_normalize

df_ev = json_normalize(
    tickets_ev,
    record_path=["lineas"],
    meta=[
        "idTicket","idEmpresa","empresaNombre","idUsuario",
        "fechaEmision","horaEmision","metodoPago",
        ["estacion","id"],["estacion","nombre"],["estacion","grupo"],["estacion","nifEmpresa"],
        ["estacion","provincia"],["estacion","municipio"],["estacion","direccion"],
        ["estacion","lat"],["estacion","lon"],["estacion","potencia_kW"],["estacion","tipo_carga"],
        ["carga","tiempo_min"],["carga","energia_kWh"],["carga","precio_kWh"],
        "baseImponibleEnergia","impuestoElectricidad","baseIVA","iva","total","moneda","tipoDocumento"
    ],
    errors="ignore"
)

df_ev = df_ev.rename(columns={
    "idTicket":"ticket_id","idEmpresa":"empresa_id","empresaNombre":"empresa_nombre",
    "idUsuario":"usuario_id","fechaEmision":"fecha","horaEmision":"hora","metodoPago":"metodo_pago",
    "estacion.id":"est_id","estacion.nombre":"est_nombre","estacion.grupo":"est_grupo","estacion.nifEmpresa":"est_nif",
    "estacion.provincia":"provincia","estacion.municipio":"municipio","estacion.direccion":"direccion",
    "estacion.lat":"lat","estacion.lon":"lon","estacion.potencia_kW":"potencia_kW","estacion.tipo_carga":"tipo_carga",
    "concepto":"concepto","energia_kWh":"energia_kWh","precioUnitario_kWh":"precio_kWh","importeEnergia":"importe_energia",
    "baseImponibleEnergia":"base_energia","impuestoElectricidad":"impuesto_electricidad","baseIVA":"base_iva","tipoDocumento":"tipo_documento",
    "carga.tiempo_min":"carga_tiempo_min","carga.energia_kWh":"carga_energia_kWh","carga.precio_kWh":"carga_precio_kWh"
})

if "precio_kwh" in df_ev.columns and "precio_kWh" not in df_ev.columns:
    df_ev = df_ev.rename(columns={"precio_kwh":"precio_kWh"})

for c in ["energia_kWh","precio_kWh","importe_energia","base_energia","impuesto_electricidad","base_iva","iva","total","lat","lon","potencia_kW","carga_energia_kWh","carga_precio_kWh"]:
    if c in df_ev.columns:
        df_ev[c] = pd.to_numeric(df_ev[c], errors="coerce")

print("Tickets EV totales:", len(df_ev))
print(df_ev.head(5))

print("\nPor empresa:")
print(df_ev.groupby("empresa_id").size())

print("\nPor tipo de carga:")
print(df_ev.groupby("tipo_carga").agg(n=("tipo_carga","size"), p_med=("precio_kWh","mean")).reset_index())

print("\nUbicación (muestra):")
cols_loc = [c for c in ["est_id","est_nombre","est_grupo","est_nif","provincia","municipio","direccion","lat","lon","potencia_kW","tipo_carga"] if c in df_ev.columns]
print(df_ev[cols_loc].head(10))


Tickets EV totales: 450
   concepto  energia_kWh  precio_kWh  importe_energia        ticket_id  \
0  Carga DC        13.07       0.383             5.01  EV-79FDD905B107   
1  Carga AC         8.94       0.420             3.75  EV-077B8D900145   
2  Carga DC        11.11       0.351             3.90  EV-B3B507866157   
3  Carga AC        10.66       0.448             4.78  EV-207218E53B70   
4  Carga DC        11.91       0.359             4.28  EV-C895BB6AA7DD   

  empresa_id        empresa_nombre   usuario_id       fecha      hora  ...  \
0     EMP001  TransporteEV_01 S.L.  EMP001-U001  2025-04-13  04:23:15  ...   
1     EMP001  TransporteEV_01 S.L.  EMP001-U001  2025-01-21  00:57:45  ...   
2     EMP001  TransporteEV_01 S.L.  EMP001-U001  2025-04-11  18:35:04  ...   
3     EMP001  TransporteEV_01 S.L.  EMP001-U001  2025-01-14  02:22:12  ...   
4     EMP001  TransporteEV_01 S.L.  EMP001-U001  2025-04-19  19:03:05  ...   

  carga_tiempo_min carga_energia_kWh carga_precio_kWh base_ene